# Linked Lists, Stacks, Queues, Heaps and Top-K

Notebook for **Chapter 9** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch09.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
samples = [("soup", 120), ("curry", 340), ("soup", 95), ("steak", 870),
           ("curry", 410), ("soup", 130), ("steak", 640), ("curry", 90)]

## 9.1 Linked Lists

In [ ]:
class Node:
    def __init__(s, v, n=None): s.val = v; s.next = n
def build(a):
    h = None
    for x in reversed(a): h = Node(x, h)
    return h
def to_list(h):
    out = []
    while h: out.append(h.val); h = h.next
    return out

In [ ]:
def reverse(h):
    prev = None
    while h:
        h.next, prev, h = prev, h, h.next
    return prev
print(to_list(reverse(build([1,2,3,4,5]))))

[5, 4, 3, 2, 1]


In [ ]:
def reverse_wrong(h):
    prev = None
    while h:
        h.next = prev
        prev = h
        h = h.next        # now this is prev, not the old next
    return prev
print(to_list(reverse_wrong(build([1, 2, 3]))))

[1]


In [ ]:
def has_cycle(h):
    s = f = h
    while f and f.next:
        s = s.next; f = f.next.next
        if s is f: return True
    return False
h = build([1,2,3,4]); print(has_cycle(h))
t = h
while t.next: t = t.next
t.next = h.next; print(has_cycle(h))

def middle(h):
    s = f = h
    while f and f.next: s = s.next; f = f.next.next
    return s.val
print(middle(build([1,2,3,4,5])), middle(build([1,2,3,4,5,6])))

False
True
3 4


In [ ]:
def merge2(a, b):
    d = cur = Node(0)                  # dummy head avoids special cases
    while a and b:
        if a.val <= b.val: cur.next = a; a = a.next
        else: cur.next = b; b = b.next
        cur = cur.next
    cur.next = a or b
    return d.next
print(to_list(merge2(build([1,4,7]), build([2,3,9,10]))))

def remove_nth_end(h, n):
    d = Node(0, h); f = s = d
    for _ in range(n): f = f.next      # give fast a head start of n
    while f.next: f = f.next; s = s.next
    s.next = s.next.next
    return d.next
print(to_list(remove_nth_end(build([1,2,3,4,5]), 2)))

[1, 2, 3, 4, 7, 9, 10]
[1, 2, 3, 5]


## 9.2 Stacks

In [ ]:
def valid(s):
    pairs = {')': '(', ']': '[', '}': '{'}; st = []
    for c in s:
        if c in '([{': st.append(c)
        elif not st or st.pop() != pairs[c]: return False
    return not st
print(valid("{[()]}"), valid("([)]"), valid("(("))

True False False


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
st = []
st.pop()

IndexError: pop from empty list


In [ ]:
def eval_rpn(tokens):
    st = []
    for t in tokens:
        if t in "+-*/" and len(t) == 1:
            b = st.pop(); a = st.pop()
            st.append(a+b if t == '+' else a-b if t == '-' else a*b if t == '*' else int(a/b))
        else: st.append(int(t))
    return st[0]
print(eval_rpn(["2","1","+","3","*"]), eval_rpn(["4","13","5","/","+"]))

9 6


In [ ]:
class MinStack:
    def __init__(s): s.st = []
    def push(s, x): s.st.append((x, min(x, s.st[-1][1]) if s.st else x))
    def pop(s): return s.st.pop()[0]
    def min(s): return s.st[-1][1]
m = MinStack()
for x in (5, 3, 7, 2): m.push(x)
print(m.min()); m.pop(); print(m.min()); m.pop(); m.pop(); print(m.min())

2
3
5


## 9.3 Queues and Deques

In [ ]:
from collections import deque
q = deque()
q.append(1); q.append(2); q.append(3)
print(q.popleft(), list(q))
d = deque([1,2,3,4,5]); d.rotate(2); print(list(d))
dm = deque(maxlen=3)
for x in range(6): dm.append(x)
print(list(dm))

1 [2, 3]
[4, 5, 1, 2, 3]
[3, 4, 5]


In [ ]:
def bfs_levels(n):
    q = deque([(1, 0)]); seen = {1}
    while q:
        v, s = q.popleft()
        if v == n: return s
        for w in (v*2, v+1):
            if w <= n and w not in seen:
                seen.add(w); q.append((w, s+1))
print(bfs_levels(10))

4


In [ ]:
import time
L = list(range(200000)); D = deque(L)
t = time.perf_counter()
for _ in range(20000): L.pop(0)
t1 = time.perf_counter() - t
t = time.perf_counter()
for _ in range(20000): D.popleft()
t2 = time.perf_counter() - t
print("list pop(0) slower than popleft by >20x:", t1 / t2 > 20)

list pop(0) slower than popleft by >20x: True


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from collections import deque
deque().popleft()

IndexError: pop from an empty deque


## 9.4 Monotonic Stack and Monotonic Queue

In [ ]:
def next_greater(a):
    res = [-1] * len(a); st = []
    for i, x in enumerate(a):
        while st and a[st[-1]] < x:
            res[st.pop()] = x
        st.append(i)
    return res
print(next_greater([2,1,2,4,3]))

[4, 2, 4, -1, -1]


In [ ]:
def daily_temps(t):
    res = [0] * len(t); st = []
    for i, x in enumerate(t):
        while st and t[st[-1]] < x:
            j = st.pop(); res[j] = i - j
        st.append(i)
    return res
print(daily_temps([73,74,75,71,69,72,76,73]))

def largest_rect(h):
    st = []; best = 0; h = h + [0]          # sentinel 0 flushes the stack
    for i, x in enumerate(h):
        start = i
        while st and st[-1][1] > x:
            idx, hh = st.pop()
            best = max(best, hh * (i - idx)); start = idx
        st.append((start, x))
    return best
print(largest_rect([2,1,5,6,2,3]))

[1, 1, 4, 2, 1, 1, 0, 0]
10


In [ ]:
def sliding_max(a, k):
    dq = deque(); out = []                  # dq holds indexes, values decreasing
    for i, x in enumerate(a):
        while dq and a[dq[-1]] <= x: dq.pop()
        dq.append(i)
        if dq[0] <= i - k: dq.popleft()     # front left the window
        if i >= k - 1: out.append(a[dq[0]])
    return out
print(sliding_max([1,3,-1,-3,5,3,6,7], 3))

[3, 3, 5, 5, 6, 7]


## 9.5 Heaps and Priority Queues

In [ ]:
import heapq
h = []
for x in [5,3,8,1,9,2]: heapq.heappush(h, x)
print(h)
print(heapq.heappop(h))
print(h)
a = [9,4,7,1,8,2]; heapq.heapify(a); print(a)
mh = []
for x in [5,3,8]: heapq.heappush(mh, -x)
print(-heapq.heappop(mh))

[1, 3, 2, 5, 9, 8]
1
[2, 3, 8, 5, 9]
[1, 4, 2, 9, 8, 7]
8


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import heapq
h = [(1, {"a": 1})]
heapq.heappush(h, (1, {"b": 2}))

TypeError: '<' not supported between instances of 'dict' and 'dict'


In [ ]:
pq = []
for pr, task in [(2,"write"), (1,"urgent"), (3,"idle"), (1,"fire")]:
    heapq.heappush(pq, (pr, task))
print([heapq.heappop(pq) for _ in range(4)])

def merge_k(lists):
    h = [(l[0], i, 0) for i, l in enumerate(lists) if l]; heapq.heapify(h); out = []
    while h:
        v, i, j = heapq.heappop(h); out.append(v)
        if j + 1 < len(lists[i]): heapq.heappush(h, (lists[i][j+1], i, j+1))
    return out
print(merge_k([[1,4,5], [1,3,4], [2,6]]))
print(list(heapq.merge([1,4,5], [1,3,4], [2,6])))

[(1, 'fire'), (1, 'urgent'), (2, 'write'), (3, 'idle')]
[1, 1, 2, 3, 4, 4, 5, 6]
[1, 1, 2, 3, 4, 4, 5, 6]


In [ ]:
class RunningMedian:
    def __init__(s): s.lo = []; s.hi = []      # lo is a max-heap (negated)
    def add(s, x):
        heapq.heappush(s.lo, -x); heapq.heappush(s.hi, -heapq.heappop(s.lo))
        if len(s.hi) > len(s.lo): heapq.heappush(s.lo, -heapq.heappop(s.hi))
    def median(s):
        return -s.lo[0] if len(s.lo) > len(s.hi) else (-s.lo[0] + s.hi[0]) / 2
r = RunningMedian()
for x in [5, 15, 1, 3]:
    r.add(x); print(x, r.median())

5 5
15 10.0
1 5
3 4.0


## 9.6 Top-K Problems

In [ ]:
from collections import Counter
def top_k_frequent(a, k):
    c = Counter(a)
    return heapq.nlargest(k, c, key=c.get)
print(top_k_frequent([1,1,1,2,2,3,4,4,4,4], 2))

[4, 1]


In [ ]:
def kth_largest(a, k):
    h = a[:k]; heapq.heapify(h)
    for x in a[k:]:
        if x > h[0]: heapq.heapreplace(h, x)
    return h[0]
print(kth_largest([3,2,1,5,6,4], 2), kth_largest([3,2,3,1,2,4,5,5,6], 4))

5 4


In [ ]:
def k_closest(pts, k):
    return heapq.nsmallest(k, pts, key=lambda p: p[0]**2 + p[1]**2)
print(k_closest([(1,3), (-2,2), (5,8), (0,1)], 2))

words = ["a","b","a","c","b","a","d"]
c = Counter(words)
print(sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))[:2])

[(0, 1), (-2, 2)]
[('a', 3), ('b', 2)]


In [ ]:
import heapq
from collections import deque
print(heapq.nlargest(3, samples, key=lambda s: s[1]))

h = []
for dish, secs in samples:
    if len(h) < 3: heapq.heappush(h, secs)
    elif secs > h[0]: heapq.heapreplace(h, secs)
print(sorted(h, reverse=True))

q = deque(samples)
print(q.popleft(), len(q))

win = deque(maxlen=3)
for dish, secs in samples: win.append(secs)
print(list(win), round(sum(win) / 3, 1))

[('steak', 870), ('steak', 640), ('curry', 410)]
[870, 640, 410]
('soup', 120) 7
[130, 640, 90] 286.7


## 9.7 Building a Heap in O(n): Why It Is Faster Than n Pushes

In [ ]:
import heapq
a = [9, 4, 7, 1, 6, 2]
heapq.heapify(a)
print(a)

[1, 4, 2, 9, 6, 7]


In [ ]:
import random

def sift_down(a, i, n, counter):
    while True:
        l, r, s = 2 * i + 1, 2 * i + 2, i
        if l < n:
            counter[0] += 1
            if a[l] < a[s]: s = l
        if r < n:
            counter[0] += 1
            if a[r] < a[s]: s = r
        if s == i: return
        a[i], a[s] = a[s], a[i]
        i = s

def sift_up(a, i, counter):
    while i > 0:
        p = (i - 1) // 2
        counter[0] += 1
        if a[i] < a[p]:
            a[i], a[p] = a[p], a[i]; i = p
        else:
            return

def build_bottom_up(data):
    a, c = list(data), [0]
    for i in range(len(a) // 2 - 1, -1, -1):
        sift_down(a, i, len(a), c)
    return a, c[0]

def build_by_pushes(data):
    a, c = [], [0]
    for x in data:
        a.append(x); sift_up(a, len(a) - 1, c)
    return a, c[0]

random.seed(7)
data = random.sample(range(10000), 1023)
h1, c1 = build_bottom_up(data)
h2, c2 = build_by_pushes(data)
ok = lambda h: all(h[(i - 1) // 2] <= h[i] for i in range(1, len(h)))
print("both valid heaps:", ok(h1), ok(h2))
print("bottom-up comparisons:", c1, "  per element:", round(c1 / 1023, 2))
print("push-by-push comparisons:", c2, "  per element:", round(c2 / 1023, 2))

both valid heaps: True True
bottom-up comparisons: 1916   per element: 1.87
push-by-push comparisons: 2319   per element: 2.27


## Chapter recap

## Practice set

**1.** What does a stack return after `push 1, push 2, push 3, pop, pop`?

In [ ]:
def reverse(h):
    prev = None
    while h:
        h.next = prev
        prev = h
        h = h.next
    return prev

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch09.html